# 🏆 SupportHR CV Industry Classifier - Kaggle Notebook Deployment

Notebook này dùng để huấn luyện và triển khai Model Pipeline phân loại ngành nghề CV (24 nhóm ngành) trên **Kaggle Notebooks**, tận dụng GPU/CPU Kaggle và kết nối trực tiếp vào API chung của SupportHR (`cv-match-api`) qua **Cloudflare Tunnel**.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q fastapi uvicorn scikit-learn joblib pycloudflared pydantic requests pandas

### Bước 2: Tự động tìm và huấn luyện trên Dataset Kaggle Resume

In [ ]:
import os
import re
import glob
from pathlib import Path
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

def clean_text(text: str) -> str:
    normalized = str(text).lower()
    normalized = re.sub(r"[^\w\s]+", " ", normalized, flags=re.UNICODE)
    normalized = normalized.replace("_", " ")
    return re.sub(r"\s+", " ", normalized).strip()

# Tìm file Resume.csv trong thư mục /kaggle/input
csv_candidates = glob.glob("/kaggle/input/**/Resume.csv", recursive=True) + glob.glob("/kaggle/input/**/*resume*.csv", recursive=True)
dataset_csv = csv_candidates[0] if csv_candidates else None

if dataset_csv and os.path.exists(dataset_csv):
    print(f"✅ Tìm thấy dataset Kaggle: {dataset_csv}")
    df = pd.read_csv(dataset_csv, on_bad_lines='skip')
    text_col = 'Resume_str' if 'Resume_str' in df.columns else ('text' if 'text' in df.columns else df.columns[0])
    label_col = 'Category' if 'Category' in df.columns else ('label' if 'label' in df.columns else df.columns[1])
    print(f"   Sử dụng cột text: {text_col}, cột label: {label_col}")
    df[text_col] = df[text_col].fillna('').map(clean_text)
    df[label_col] = df[label_col].fillna('').astype(str).str.strip().str.upper()
    df = df[df[text_col] != '']
    X, y = df[text_col], df[label_col]
    print(f"   Huấn luyện trên {len(df)} dòng dữ liệu thật, số nhãn: {df[label_col].nunique()}...")
else:
    print("ℹ️ Chưa gắn dataset Kaggle, khởi tạo bộ nhãn tiêu chuẩn 24 categories SupportHR...")
    seed_data = [
        ("accounting finance ledger tax audit balance sheet accountant bookkeeping reconciliation", "ACCOUNTANT"),
        ("legal advocate court lawyer litigation litigation counsel compliance contract law", "ADVOCATE"),
        ("agriculture crop farming agronomy harvesting soil seeds livestock irrigation", "AGRICULTURE"),
        ("apparel fashion textile garment clothing designer merchandising pattern fabrication", "APPAREL"),
        ("fine arts painting sculpture gallery illustrator artist visual creative exhibition", "ARTS"),
        ("automobile vehicle automotive mechanic engine car repair transport diagnostic maintenance", "AUTOMOBILE"),
        ("aviation pilot flight aircraft airline aeronautical aerospace crew navigation avionics", "AVIATION"),
        ("banking loan credit deposit investment branch banking banker mortgage retail banking", "BANKING"),
        ("bpo customer support call center outsourcing telemarketing agent technical support inbound", "BPO"),
        ("business development partnership growth strategy b2b sales lead enterprise account", "BUSINESS-DEVELOPMENT"),
        ("chef culinary cooking cuisine kitchen restaurant pastry food recipe hospitality menu", "CHEF"),
        ("civil engineering construction architect building site contractor structural inspection", "CONSTRUCTION"),
        ("consulting advisory management consultant strategy operational transformation roadmap", "CONSULTANT"),
        ("graphic designer ui ux figma product layout typography visual wireframing prototyping", "DESIGNER"),
        ("digital media social content video creator broadcast marketing journalism podcast", "DIGITAL-MEDIA"),
        ("electrical mechanical engineering firmware hardware maintenance cad plc electronics", "ENGINEERING"),
        ("financial analyst equity wealth investment banking portfolio risk valuation treasury", "FINANCE"),
        ("fitness trainer gym coach exercise workout health sports athletics personal training", "FITNESS"),
        ("doctor nurse medical hospital clinical patient healthcare therapy pharmacy physician", "HEALTHCARE"),
        ("human resources talent acquisition recruitment payroll hr screening onboarding employee relations", "HR"),
        ("python fastapi react typescript software engineer developer backend frontend cloud devops fullstack", "INFORMATION-TECHNOLOGY"),
        ("public relations communications press release media spokesperson crisis branding communications", "PUBLIC-RELATIONS"),
        ("sales executive quota pipeline negotiation account manager cold calling b2b revenue", "SALES"),
        ("teacher tutor education curriculum classroom school professor pedagogy lecturing academic", "TEACHER"),
    ]
    X = [d[0] for d in seed_data]
    y = [d[1] for d in seed_data]

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=15000)),
    ("clf", LinearSVC(random_state=42, dual="auto")),
])
pipeline.fit(X, y)
model_path = "/kaggle/working/text_classifier_model.pkl"
joblib.dump(pipeline, model_path)
print(f"✅ Huấn luyện hoàn tất! Đã lưu model vào: {model_path}")


### Bước 3: Tạo và Khởi động FastAPI Server

In [ ]:
%%writefile /kaggle/working/server.py
import os, re, math
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib

app = FastAPI(title="SupportHR Kaggle Classifier")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

_model = None
_model_source = os.getenv("MODEL_SOURCE", "kaggle://resume-classifier-v1")

class ClassifyRequest(BaseModel):
    cv_text: str = Field(...)
    top_k: int = Field(default=3, ge=1, le=24)

def clean_text(text: str) -> str:
    normalized = str(text).lower()
    normalized = re.sub(r"[^\w\s]+", " ", normalized, flags=re.UNICODE)
    normalized = normalized.replace("_", " ")
    return re.sub(r"\s+", " ", normalized).strip()

def _softmax(values: list[float]) -> list[float]:
    if not values: return []
    m = max(values)
    exps = [math.exp(v - m) for v in values]
    s = sum(exps)
    return [v / s for v in exps] if s > 0 else [0.0 for _ in values]

@app.on_event("startup")
def startup():
    global _model
    p = Path("/kaggle/working/text_classifier_model.pkl")
    if p.is_file():
        _model = joblib.load(p)
        print("Kaggle Model loaded successfully.")

@app.get("/health")
def health():
    return {"status": "ok", "ready": _model is not None}

@app.get("/api/cv/classifier-status")
@app.get("/api/classifier-status")
def status():
    classes = [str(c) for c in getattr(_model, "classes_", [])] if _model else []
    return {
        "ready": _model is not None,
        "model_source": _model_source,
        "label_count": len(classes),
        "labels": classes,
        "error": None if _model else "Model not ready"
    }

@app.post("/api/cv/classify-industry")
@app.post("/api/classify-industry")
def classify(payload: ClassifyRequest):
    if not _model:
        raise HTTPException(503, "Model not ready")
    cleaned = clean_text(payload.cv_text)
    if not cleaned:
        raise HTTPException(400, "Empty CV text")
    classes = [str(c) for c in getattr(_model, "classes_", [])]
    if hasattr(_model, "decision_function"):
        scores = _model.decision_function([cleaned])
        if hasattr(scores, "tolist"): scores = scores.tolist()
        if isinstance(scores, list) and scores and isinstance(scores[0], list): scores = scores[0]
        if not isinstance(scores, list): scores = [float(scores)]
        probs = _softmax([float(v) for v in scores])
    elif hasattr(_model, "predict_proba"):
        probs = _model.predict_proba([cleaned])[0]
    else:
        probs = [1.0 if c == _model.predict([cleaned])[0] else 0.0 for c in classes]
    ranked = sorted([{"label": l, "score": round(float(s), 4)} for l, s in zip(classes, probs)], key=lambda x: x["score"], reverse=True)[:payload.top_k]
    return {
        "predicted_label": ranked[0]["label"] if ranked else "UNKNOWN",
        "confidence": ranked[0]["score"] if ranked else 0.0,
        "top_predictions": ranked,
        "model_source": _model_source
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


### Bước 4: Mở Cloudflare Tunnel và Lấy URL Kết Nối

In [ ]:
import subprocess, time, requests
from pycloudflared import try_cloudflare

server_process = subprocess.Popen(["python", "/kaggle/working/server.py"])
print("⏳ Đang khởi động FastAPI server trên Kaggle...")
time.sleep(3)

tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.rstrip("/")

print("\n" + "="*60)
print("✅ KAGGLE SERVER ĐÃ SẴN SÀNG!")
print(f"🌐 Public HTTPS URL: {public_url}")
print("="*60)

try:
    res = requests.get(f"{public_url}/api/cv/classifier-status", timeout=10)
    print(f"Trạng thái model: {res.json()}")
except Exception as e:
    print(f"Warning test tunnel: {e}")

print("\n📝 Cấu hình dán vào file api_server/.env của SupportHR:")
print(f"LOCAL_CLASSIFIER_MODE=auto")
print(f"LOCAL_CLASSIFIER_REMOTE_CLASSIFY_URL={public_url}/api/cv/classify-industry")
print(f"LOCAL_CLASSIFIER_REMOTE_STATUS_URL={public_url}/api/cv/classifier-status")
print(f"LOCAL_CLASSIFIER_REMOTE_TIMEOUT_SECONDS=10.0")
